# 01. Load and label

Goal of this notebook: connect to the DuckDB database, confirm the three tables loaded correctly,
and build the **label table** that every later model will be scored against.

The database was built by `scripts/build_database.py`. It holds:

| table | grain | rows |
|---|---|---|
| `provider` | one row per NPI, 2024 totals | ~1.30M |
| `provider_service` | one row per NPI x HCPCS code x place of service, 2024 | ~9.78M |
| `leie` | one row per OIG exclusion | ~84K |

DuckDB is a columnar SQL engine that runs in-process, like SQLite, but is built for analytics
over large tables. The 9.8M row table aggregates in seconds. We do the heavy grouping in SQL
and only pull the result into pandas.

In [1]:
import duckdb
import pandas as pd

DB_PATH = "C:/Users/palla/OneDrive/Documents/Coding Projects/Medicare Fraud ML/database/medicare_fraud.duckdb"
con = duckdb.connect(DB_PATH)

def q(sql: str) -> pd.DataFrame:
    """Run a query and return a DataFrame."""
    return con.sql(sql).df()

q("SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables() ORDER BY 1")

,table_name,approx_rows
0,leie,83842
1,provider,1296739
2,provider_service,9781673


## 1. What a provider row looks like

The `provider` table already has totals per NPI (services, beneficiaries, payments) plus
beneficiary demographics and chronic-condition percentages. It is the natural starting point
for provider-level features.

In [2]:
q("SELECT * FROM provider LIMIT 3").T

,0,1,2
Rndrng_NPI,1003000126,1003000134,1003000142
Rndrng_Prvdr_Last_Org_Name,Enkeshafi,Cibull,Khalil
Rndrng_Prvdr_First_Name,Ardalan,Thomas,Rashid
Rndrng_Prvdr_MI,NaN,L,NaN
Rndrng_Prvdr_Crdntls,M.D.,M.D.,M.D.
...,...,...,...
Bene_CC_PH_Osteoporosis_V2_Pct,18,15,17
Bene_CC_PH_Parkinson_V2_Pct,4,2,<NA>
Bene_CC_PH_Arthritis_V2_Pct,63,47,75
Bene_CC_PH_Stroke_TIA_V2_Pct,21,6,11


In [3]:
q("""
SELECT Rndrng_Prvdr_Type AS provider_type,
       count(*)                          AS n_providers,
       round(median(Tot_Srvcs))          AS median_services,
       round(median(Tot_Mdcr_Pymt_Amt))  AS median_payment
FROM provider
GROUP BY 1
ORDER BY n_providers DESC
LIMIT 15
""")

,provider_type,n_providers,median_services,median_payment
0,Nurse Practitioner,203212,253.0,13301.0
1,Physician Assistant,112514,235.0,12626.0
2,Internal Medicine,93627,623.0,44569.0
3,Family Practice,83639,568.0,31326.0
4,Physical Therapist in Private Practice,77901,1571.0,32377.0
5,Certified Registered Nurse Anesthetist (CRNA),53495,109.0,11828.0
6,Emergency Medicine,50101,330.0,31330.0
7,Anesthesiology,41082,194.0,22889.0
8,Diagnostic Radiology,32843,2949.0,82012.0
9,Chiropractic,32394,370.0,9376.0


## 2. The exclusion list, and why most of it cannot be used

The LEIE has ~84K rows but only ~8.8K carry a real NPI. The rest are `0000000000`,
mostly businesses and exclusions that predate the NPI system. Only rows with a real NPI can
join to billing data.

`EXCLTYPE` is the statutory section the exclusion was issued under. The ones that matter here:

| code | meaning |
|---|---|
| 1128a1 | felony conviction, program-related crime (Medicare/Medicaid fraud) |
| 1128a2 | felony conviction, patient abuse or neglect |
| 1128a3 | felony conviction, healthcare fraud (any payer) |
| 1128a4 | felony conviction, controlled substances |
| 1128b4 | license revoked or surrendered (often not fraud) |
| 1128b7 | fraud, kickbacks, other prohibited activities (civil) |

Section (a) exclusions are **mandatory** and follow a criminal conviction. Section (b) exclusions
are **permissive**, and 1128b4 in particular is usually a licensing action, not a fraud finding.

In [4]:
q("""
SELECT EXCLTYPE, count(*) AS n
FROM leie
WHERE NPI <> '0000000000'
GROUP BY 1 ORDER BY n DESC
LIMIT 12
""")

,EXCLTYPE,n
0,1128a1,3379
1,1128b4,2636
2,1128a4,1022
3,1128a3,681
4,1128a2,402
5,1128b14,196
6,1128b7,194
7,1128b8,96
8,1128b5,85
9,1128b1,48


## 3. Joining the two

How many providers who billed Medicare in 2024 are on the exclusion list at all?
And when were they excluded relative to the billing year?

The timing question matters. A provider excluded in 2019 is barred from billing, so they will
not be in the 2024 file. The providers we *can* see are the ones excluded during or after 2024,
whose 2024 billing therefore happened while the conduct was still undetected.
That is precisely the population a fraud model is supposed to find.

In [5]:
q("""
SELECT substr(l.EXCLDATE, 1, 4) AS exclusion_year,
       l.EXCLTYPE,
       count(DISTINCT p.Rndrng_NPI) AS n_providers
FROM provider p
JOIN leie l ON CAST(p.Rndrng_NPI AS VARCHAR) = l.NPI
GROUP BY 1, 2
ORDER BY 1, 3 DESC
""")

,exclusion_year,EXCLTYPE,n_providers
0,2015,1128a1,1
1,2024,1128a1,4
2,2024,1128a4,3
3,2024,1128b1,2
4,2024,1128b7,2
5,2024,1128b4,2
6,2024,1128b3,1
7,2025,1128b4,24
8,2025,1128a1,21
9,2025,1128a4,5


## 4. Defining the label

Decision: a provider is a **positive** if they have a section (a) mandatory exclusion
(1128a1, a2, a3, a4) or a 1128b7 fraud exclusion, dated 2024 or later.

License revocations (1128b4) are excluded from the positive class *and* from the negative class.
They are ambiguous, so we drop those providers entirely rather than guess.

This gives a very small positive class against ~1.3M negatives. That is the real shape of the
problem in program integrity work, and it is why accuracy is useless here and why the
evaluation notebook leans on ranking metrics (ROC AUC, PR AUC, precision at k).

The `labels` table is written back into the database so every later notebook uses the same definition.

In [6]:
con.execute("""
CREATE OR REPLACE TABLE labels AS
WITH excl AS (
    SELECT NPI,
           min(EXCLDATE) AS first_excl_date,
           bool_or(EXCLTYPE IN ('1128a1','1128a2','1128a3','1128a4','1128b7')) AS is_fraud_type,
           bool_or(EXCLTYPE = '1128b4') AS is_license_type
    FROM leie
    WHERE NPI <> '0000000000'
    GROUP BY NPI
)
SELECT p.Rndrng_NPI AS npi,
       CASE
         WHEN e.NPI IS NULL THEN 0                                   -- never excluded
         WHEN e.is_fraud_type AND e.first_excl_date >= '20240101' THEN 1
         ELSE NULL                                                   -- ambiguous: drop from modeling
       END AS label,
       e.first_excl_date,
       e.is_fraud_type,
       e.is_license_type
FROM provider p
LEFT JOIN excl e ON CAST(p.Rndrng_NPI AS VARCHAR) = e.NPI
""")

q("""
SELECT label, count(*) AS n
FROM labels
GROUP BY 1 ORDER BY 1 NULLS LAST
""")

,label,n
0,0,1296590
1,1,68
2,<NA>,81


## 5. Sanity check: do the positives look different at all?

Before any modeling, compare medians. If excluded providers look identical to everyone else on
raw totals, the features will have to come from billing *mix* (the `provider_service` table),
not from volume.

In [7]:
q("""
SELECT lb.label,
       count(*)                                        AS n,
       round(median(p.Tot_Benes))                      AS med_benes,
       round(median(p.Tot_Srvcs))                      AS med_services,
       round(median(p.Tot_Srvcs / p.Tot_Benes), 1)     AS med_services_per_bene,
       round(median(p.Tot_Mdcr_Pymt_Amt))              AS med_payment,
       round(median(p.Tot_HCPCS_Cds))                  AS med_distinct_codes
FROM labels lb
JOIN provider p ON p.Rndrng_NPI = lb.npi
WHERE lb.label IS NOT NULL
GROUP BY 1 ORDER BY 1
""")

,label,n,med_benes,med_services,med_services_per_bene,med_payment,med_distinct_codes
0,0,1296590,130.0,433.0,2.9,26514.0,16.0
1,1,68,117.0,425.0,5.1,24417.0,15.0


In [8]:
con.close()